# **仿真分析**

本节为msProf工具仿真数据抓取分析介绍章节，完成本章节内容的学习可以掌握如何使用profiling工具分析算子仿真数据性能瓶颈。我们将按照以下结构，带你学习msProf工具仿真数据的抓取与分析：
- 环境准备
- 如何使用msProf工具采集仿真性能数据
- 如何分析仿真性能数据
- 针对性优化
- 课后实践

---

## **1 环境准备**
本文所有内容均存放于Sources文件夹。
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。保证能正常导入代码以及正常使用工具。

In [ ]:
!mkdir -p Sources/08.03

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

---
## **2 如何使用msProf工具采集仿真性能数据**
上节课我们讲过如何使用msProf采集上板性能数据，相对于上板，仿真模式多了一个参数simulator。另外，当使用仿真模式时，需要指定仿真的产品型号并在算子工程的op_kernel的CMakeLists.txt中增加允许调试选项。所以仿真性能采集分为3步：  

1. 修改算子工程编译选项
2. 准备调用程序
3. 采集仿真数据

这里仍使用以前课程的AddCustomTemplate算子和其调用程序，我们先复制已准备好的AddCustomTemplate算子工程和测试程序，测试输入为[8, 2048]的half类型数据。

In [ ]:
# 清理已存在的custom_op
!rm -rf Sources/08.03/custom_op

# 复制之前课程的算子工程, 并修改算子工程内CANN包路径配置为实际路径
!cp -r src/custom_op Sources/08.03/custom_op;sed -i '/"ASCEND_CANN_PACKAGE_PATH": {/,/}/ s|\s*"value": ".*"| "value": "'"$ASCEND_TOOLKIT_HOME"'"|' Sources/08.03/custom_op/CMakePresets.json

# 清除已有的测试代码
!rm -rf Sources/08.03/test;

# 复制已有的测试代码
!cp -r src/custom_op/test Sources/08.03/test;



算子工程目录结构为：
```
custom_op/
├── framework/
├── op_host/
│   ├── add_custom_template.cpp          // 包含算子原型注册、tiling实现 shape与Dtype推导内容
│   └── CMakeLists.txt                   // host侧CMakeLists文件，一般不需要修改
├── op_kernel/
│   ├── add_custom_template_tiling.h     // 算子tiling定义文件   
│   ├── add_custom_template.cpp          // 算子代码实现文件 
│   └── CMakeLists.txt                   // kernel侧CMakeLists文件，一般不需要修改
├── CMakeLists.txt                       // 根目录CMakeLists文件，一般不需要修改
├── CMakePresets.json                    // CMake编译配置文件
└── build.sh                             // 编译脚本
```  

我们需要修改op_kernel/CMakeLists.txt，在其末尾增加一行运行调试的编译选项。  
```
npu_op_kernel_options(ascendc_kernels ALL OPTIONS -g)
```

In [ ]:
%%writefile -a Sources/08.03/custom_op/op_kernel/CMakeLists.txt
npu_op_kernel_options(ascendc_kernels ALL OPTIONS -g)

然后根据仿真产品型号设置仿真相关的环境变量，这里以仿真Atlas A2训练产品为例，应设置环境变量为：
```shell
export LD_LIBRARY_PATH=${ASCEND_TOOLKIT_HOME}/tools/simulator/Ascend910B1/lib:$LD_LIBRARY_PATH 
```
设置环境变量后，即可使用msprof op simulator抓取可执行程序执行的仿真性能，命令为:
```shell
msprof op simulator --output=./output_data  ./xxxx
```  
如果不设置LD_LIBRARY_PATH，也可通过增加--soc-version指定要仿真的产品型号，命令为:
```shell
msprof op simulator --soc-version=Ascend910B1 --output=./output_data ./xxxx
```  
如果确认核间数据为均匀分布，或者只想获取指定核的仿真数据，可以通过--core-id来指定核。以采集id为0的核的仿真性能为例，命令为：
```shell
msprof op simulator --soc-version=Ascend910B1 --output=./output_data --core-id=0 ./xxxx
```  

让我们执行以下命令编译部署修改CMakeLists.txt后的算子工程和调用程序，尝试采集id为0的仿真性能数据。

In [ ]:
# 清除可能存在的性能文件
!rm -rf Sources/08.03/prof

# 创建性能文件存放目录
!mkdir -p Sources/08.03/prof

# 编译部署算子
!cd Sources/08.03/custom_op;bash build.sh;./build_out/custom_opp*.run  --install-path=${HOME}/
# 编译调用代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/08.03/test/main.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/08.03/execute_op;

# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;msprof op simulator --soc-version=Ascend910B1 --output=./Sources/08.03/prof --core-id=0 ./Sources/08.03/execute_op

命令执行后，Sources/08.03/prof目录下会生成性能数据文件，目录结构如下:

```
OPPROF_20260310020803_BJNTOKUSUZMIESZL/
├── simulator/
│   ├── core0.veccore0/                   // 按照core*.veccore*或core*.cubecore*目录存放各核的数据文件
│   ├── trace.json                       // 全部核或指定核的仿真指令流水图文件
│   └── visualize_data.bin               // 全部核或指定核的仿真指令流水图文件
└── dump/                                  // 存放过程件的文件夹，无需关注
    ├── aicore_binary.o
    ├── object_dump.txt
    └── pc_start_addr.txt
```


执行以下命令查看Sources/08.03/prof查看到采集的性能数据文件。

In [ ]:
!cd Sources/08.03/prof; find . -maxdepth 3 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

---
## **3 如何分析仿真性能数据**

已抓取的仿真数据文件trace.json、visualize_data.bin通常不能直接分析性能，这里我们需要借助Chrome浏览器的chrome://tracing打开trace.json或者使用[MindStudio Insight工具](https://www.hiascend.com/document/detail/zh/mindstudio/830/GUI_baseddevelopmenttool/msascendinsightug/Insight_userguide_0027.html)打开trace.json、visualize_data.bin。 

### **3.1 Chrome浏览器打开**
在Chrome浏览器中输入“chrome://tracing”地址，并将通过msprof op simulator生成指令流水图文件（trace.json）拖到空白处打开，键盘上输入快捷键（**W：放大，S：缩小，A：左移，D：右移**）可进行查看看各个流水任务耗时的信息。例如刚刚我们抓取的AddCustomTemplate算子性能文件打开会如下：  

<img src="./images/trace_info.png"  alt="trace"  width="700px">    

我们可以比较直观的看到算子整体耗时中MTE2、VECTOR、MET3耗时均很短，而SCALAR运算耗时很长，点击右侧SCALAR可在下方看到该流水任务对应的具体代码行数。
执行以下代码下载我们抓取到的trace.json文件，尝试使用Chrome浏览器查看一下吧。

In [ ]:
import glob,base64;from IPython.display import display,HTML
f=glob.glob("Sources/08.03/prof/OPPROF_*/simulator/trace.json")
if f:
    b64=base64.b64encode(open(f[0],'rb').read()).decode()
    display(HTML(f'<a href="data:application/json;base64,{b64}" download="trace.json" style="color:white;background:#007bff;padding:5px 10px;text-decoration:none;border-radius:3px;">📥 下载</a>'))

也可以直接执行下面代码查看性能文件，该代码简单生成了一个图表模拟了chrome://tracing的展示效果，但是不可以缩放和拖动。

In [ ]:
import matplotlib.pyplot as plt, warnings, json, pandas as pd, os, glob
plt.set_loglevel("error"); warnings.filterwarnings('ignore', category=UserWarning)

BASE_DIR = "Sources/08.03/prof"
try:
    trace_path = f"{sorted(glob.glob(f'{BASE_DIR}/OPPROF*'), key=os.path.getctime)[-1]}/simulator/trace.json"
except:
    trace_path = "Sources/08.03/prof/OPPROF_20260310110753_XXSVYUAHKSEHAZXS/simulator/trace.json"

df = pd.DataFrame([{"name":e.get("name","unknown"),"start":e["ts"]/1e6,"dur":e["dur"]/1e6,"thread":f"Thread-{e.get('tid',0)}"} 
                   for e in json.load(open(trace_path))["traceEvents"] if "ts" in e and "dur" in e])
plt.figure(figsize=(12, 5))

for thread in df["thread"].unique():
    t_df = df[df["thread"] == thread]
    plt.barh(thread, t_df["dur"], left=t_df["start"], label=t_df["name"].iloc[0], alpha=0.7)

thread_names = [th.replace("Thread-", "") for th in df["thread"].unique()]
plt.yticks(ticks=df["thread"].unique(), labels=thread_names)

plt.xlabel("Time (ms)"); plt.title("Trace.json")
plt.legend(loc="upper right", bbox_to_anchor=(1.2, 1)); plt.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

### **3.2 使用MindStudio Insight打开**
若要使用MindStudio Insight进行查看时，需要单独安装MindStudio Insight软件包，具体下载链接请参见[安装与卸载](https://www.hiascend.com/document/detail/zh/mindstudio/830/GUI_baseddevelopmenttool/msascendinsightug/Insight_userguide_0006.html)。  
安装好MindStudio Insight软件包后，我们可以在工具中导入刚刚抓取到的性能数据文件。   

<img src="./images/chose_file.png" alt="trace"  width="350px">

导入后，时间线界面与Chrome浏览器中打开效果一致，如图:   

<img src="./images/timeline.png" alt="trace"  width="700px">

切换到源码界面，我们在设置好对应源码后，即可查看每行代码对应的时钟周期，可以更直观的看到耗时较多的代码或代码块。  

<img src="./images/code_info.png" alt="trace"  width="700px">

---
## **4 针对性优化**
根据trace图或者源码时钟周期我们可以看出耗时主要集中在printf打印，所以推测将printf打印屏蔽后，算子的性能会有较大提升。尝试根据分析修改以下代码：

In [ ]:
%%writefile Sources/08.03/custom_op/op_kernel/add_custom_template.cpp 
#include "kernel_operator.h"
#include "add_custom_template_tiling.h"
#include "kernel_operator_dump_tensor_intf_impl.h"
constexpr int32_t BUFFER_NUM = 1;

template <class dtypeX, class dtypeY, class dtypeZ>
class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
        xGm.SetGlobalBuffer((__gm__ dtypeX *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        yGm.SetGlobalBuffer((__gm__ dtypeY *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        zGm.SetGlobalBuffer((__gm__ dtypeZ *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(dtypeX));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(dtypeY));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(dtypeZ));
    }

    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
        AscendC::printf("Core %d executed %d times in total\n",  AscendC::GetBlockIdx(), loopCount);
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.AllocTensor<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.AllocTensor<dtypeY>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.DeQue<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.DeQue<dtypeY>();
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.AllocTensor<dtypeZ>();
        AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
        outQueueZ.EnQue<dtypeZ>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.DeQue<dtypeZ>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<dtypeX> xGm;
    AscendC::GlobalTensor<dtypeY> yGm;
    AscendC::GlobalTensor<dtypeZ> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

 __global__ __aicore__ void add_custom_template(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling)
{
    REGISTER_TILING_DEFAULT(TilingDataTemplate);
    GET_TILING_DATA_WITH_STRUCT(TilingDataTemplate, tiling_data, tiling);
    KernelAdd<DTYPE_X, DTYPE_Y, DTYPE_Z> op;
    op.Init(x, y, z, tiling_data.totalLength, tiling_data.tileNum);
    op.Process();
}

修改完成后重新部署算子并抓取性能：

In [ ]:
# 编译部署算子
!cd Sources/08.03/custom_op;bash build.sh;./build_out/custom_opp*.run  --install-path=${HOME}/

# 清除可能存在的性能文件
!rm -rf Sources/08.03/prof2
# 创建性能文件存放目录
!mkdir -p Sources/08.03/prof2
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;msprof op simulator --soc-version=Ascend910B1 --output=./Sources/08.03/prof2 --core-id=0 ./Sources/08.03/execute_op

抓取仿真性能后，使用Chrome浏览器读取性能数据与之前对比，观察是否符合预期，减少了scalar运算耗时。

In [ ]:
import glob,base64;from IPython.display import display,HTML
f=glob.glob("Sources/08.03/prof2/OPPROF_*/simulator/trace.json")
if f:
    b64=base64.b64encode(open(f[0],'rb').read()).decode()
    display(HTML(f'<a href="data:application/json;base64,{b64}" download="trace.json" style="color:white;background:#007bff;padding:5px 10px;text-decoration:none;border-radius:3px;">📥 下载</a>'))

预期如图：  

<img src="./images/no_print_trace.png" alt="prof_no_printf"  width="700px">


或执行下面代码简单查看

In [ ]:
import matplotlib.pyplot as plt, warnings, json, pandas as pd, os, glob
plt.set_loglevel("error"); warnings.filterwarnings('ignore', category=UserWarning)

BASE_DIR = "Sources/08.03/prof2"
try:
    trace_path = f"{sorted(glob.glob(f'{BASE_DIR}/OPPROF*'), key=os.path.getctime)[-1]}/simulator/trace.json"
except:
    trace_path = "Sources/08.03/prof2/OPPROF_20260310110753_XXSVYUAHKSEHAZXS/simulator/trace.json"

df = pd.DataFrame([{"name":e.get("name","unknown"),"start":e["ts"]/1e6,"dur":e["dur"]/1e6,"thread":f"Thread-{e.get('tid',0)}"} 
                   for e in json.load(open(trace_path))["traceEvents"] if "ts" in e and "dur" in e])
plt.figure(figsize=(12, 5))

for thread in df["thread"].unique():
    t_df = df[df["thread"] == thread]
    plt.barh(thread, t_df["dur"], left=t_df["start"], label=t_df["name"].iloc[0], alpha=0.7)

thread_names = [th.replace("Thread-", "") for th in df["thread"].unique()]
plt.yticks(ticks=df["thread"].unique(), labels=thread_names)

plt.xlabel("Time (ms)"); plt.title("Trace.json")
plt.legend(loc="upper right", bbox_to_anchor=(1.2, 1)); plt.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

---
## **课后实践**
请根据本节课程学习内容完成以下题目进行自测。

1. （单选题）在AscendC算子仿真分析环节，原文档中提到，开展算子性能仿真验证、获取性能数据的前置条件是？   
    A. 完成算子开发并能正常调用   
    B. 完成算子Kerne实现逻辑即可采集性能数据  
    C. 仅通过编译器静态分析代码逻辑，即可分析出性能数据，不需要借助工具   
    D. 手动计算理论性能值，不需要借助工具  

2. （单选题）Ascend C算子仿真数据抓取完成后，主要通过查看哪类核心文件来获取详细的性能耗时、单元利用率等关键性能数据？  
    A. 源码编译生成的.o目标文件  
    B. 仿真运行生成的性能统计trace.json和visualize_data.bin文件  
    C. 算子定义头文件  
    D. 系统通用日志文件  

3. （单选题）AscendC算子仿真性能分析时，重点关注的核心性能指标不包含以下哪一项？    
    A. 算子整体执行耗时  
    B. AI Core计算单元利用率  
    C. 数据搬运带宽与搬运耗时占比  
    D. 操作系统版本号  

4. （单选题）Ascend C算子仿真后的性能瓶颈定位，可行的方法是？  
    A. 随机修改代码尝试优化  
    B. 对比仿真报告中计算模块、搬运模块、同步等待的耗时占比，定位核心瓶颈  
    C. 仅查看代码行数判断效率  
    D. 依赖硬件自带的自动瓶颈提示  

5. （单选题）Ascend C仿真分析相比硬件实测，在算子性能优化阶段的核心优势是？  
    A. 无需依赖算子实际运行的实体昇腾硬件，借助仿真模拟实际硬件行为完成性能验证与瓶颈定位，缩短优化周期  
    B. 性能数据比硬件实测更精准  
    C. 可以直接修改硬件底层参数  
    D. 不需要编写任何算子代码  

执行以下代码查看答案：

In [ ]:
!cat ./answer/08.03_answer.txt